## Definir agente

In [1]:
from langchain_openai import ChatOpenAI

# 1. Initialize the local LM Studio model
local_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",  # Points to LM Studio
    api_key="lm-studio",                 # LM Studio doesn't require a real key, but a placeholder string prevents errors
    model="google/gemma-4-12b-qat",      # Match the name/identifier loaded in LM Studio
    temperature=0                       # Low temperature is critical for agent tool accuracy
)

In [2]:
from deepagents import create_deep_agent
from pathlib import Path

# Importar api key desde las variables de entorno
from dotenv import load_dotenv
load_dotenv()

# importar tools (relative import for running from this directory)
from tools import download_viirs_snpp_peru, fires_clean, download_ctry_shp, intersect_and_collapse_fires_peru, folium_choropleth_peru

system_prompt = """You are a geospatial analyst for Peru wildfire data. 

When asked to analyze fires, load and follow the peru-fire-analysis skill.
Report statistics after aggregation and provide output file paths."""

# Get the directory where this script lives (for skills path)
SCRIPT_DIR = Path("/Users/jgastanaduy/GitHub/geo-agent/deepagents/clase2")

# Definir un agente con el modelo Gemini 3.5 Flash de Google GenAI
agent = create_deep_agent(
    model= local_llm,
    tools=[download_viirs_snpp_peru, fires_clean, download_ctry_shp, 
           intersect_and_collapse_fires_peru, folium_choropleth_peru],
    system_prompt=system_prompt,
    skills=[str(SCRIPT_DIR)]  # Points to folder with SKILL.md
)

## Probar agente

In [3]:
query = """ Download and analyze forestal fires in Peru for 2024.
I want to see which departments had the most fires and generate an HTML map."""

In [4]:
# Run the agent
result = agent.invoke(
        {"messages": [{"role": "user", 
                       "content": query}]}
)

In [5]:
# Usar librería rich para imprimir el resultado de manera más legible
from rich import print as rprint
rprint(result)

{
    'messages': [
        HumanMessage(
            content=' Download and analyze forestal fires in Peru for 2024.\nI want to see which departments had 
the most fires and generate an HTML map.',
            additional_kwargs={},
            response_metadata={},
            id='671a77de-c831-444b-9172-544e6b2bbce6'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 329,
                    'prompt_tokens': 6895,
                    'total_tokens': 7224,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 209,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'google/gemma-4-12b-qat',
                'system_fingerprint': 'google/gemma-4-12b-qat',
                'id': 'chatcmpl-6yod16fat69awcy1ag6l0j',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ea376-2cd5-7a23-8b96-a8746bd683a6-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {
                                'content': 'Download VIIRS-SNPP fire data for Peru for 2024',
                                'status': 'in_progress'
                            },
                            {'content': 'Clean the downloaded fire data', 'status': 'pending'},
                            {'content': 'Download Peru administrative boundary polygons', 'status': 'pending'},
                            {
                                'content': 'Intersect fire points with administrative polygons and aggregate by 
department',
                                'status': 'pending'
                            },
                            {
                                'content': 'Generate an interactive choropleth map and provide the output path',
                                'status': 'pending'
                            }
                        ]
                    },
                    'id': '179276990',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 6895,
                'output_tokens': 329,
                'total_tokens': 7224,
                'input_token_details': {},
                'output_token_details': {'reasoning': 209}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': 'Download VIIRS-SNPP fire data for Peru for 2024', 'status':
'in_progress'}, {'content': 'Clean the downloaded fire data', 'status': 'pending'}, {'content': 'Download Peru 
administrative boundary polygons', 'status': 'pending'}, {'content': 'Intersect fire points with administrative 
polygons and aggregate by department', 'status': 'pending'}, {'content': 'Generate an interactive choropleth map 
and provide the output path', 'status': 'pending'}]",
            name='write_todos',
            id='5025f8d0-1edd-4dfa-93f6-ba21c8c90e24',
            tool_call_id='179276990'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 27,
                    'prompt_tokens': 7134,
                    'total_tokens': 7161,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasonin